In [0]:
from pyspark.sql.functions import *

SOURCE_TABLE = "fuel_project_dev.silver.google_reviews"
TARGET_TABLE = "fuel_project_dev.gold.fact_google_reviews"
CHECKPOINT_LOCATION = "/Volumes/fuel_project_dev/checkpoints/gold/fact_google_reviews"

In [0]:
# Create target table if not exists
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {TARGET_TABLE} (
        review_id STRING,
        customer_id INT,
        fuel_station_id STRING,
        rating INT,
        review STRING,
        sentiment STRING,
        is_review_missing INT,
        review_length INT,
        date_key STRING,
        modified_ts TIMESTAMP,
        processed_at TIMESTAMP
    )
    USING DELTA
    CLUSTER BY (fuel_station_id, date_key)
    TBLPROPERTIES (
        delta.autoOptimize.optimizeWrite = true,
        delta.autoOptimize.autoCompact = true
    )
""")



In [0]:
# Read streaming data
reviews_stream = (
    spark.readStream
    .format("delta")
    .table(SOURCE_TABLE)
)

In [0]:

# Transform
fact_stream = (
    reviews_stream
    .select(
        col("review_id"),
        col("customer_id"),
        col("fuel_station_id"),
        col("rating"),
        col("review"),
        when(col("rating") >= 4, "Positive")
        .when(col("rating") == 3, "Neutral")
        .otherwise("Negative").alias("sentiment"),
        when(col("review").isNull(), 1).otherwise(0).alias("is_review_missing"),
        length(col("review")).alias("review_length"),
        date_format(col("modified_ts"), "yyyyMMdd").alias("date_key"),
        col("modified_ts"),
        current_timestamp().alias("processed_at")
    )
)



In [0]:
# Write stream
query = (
    fact_stream.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", CHECKPOINT_LOCATION)
    .trigger(availableNow=True)
    .table(TARGET_TABLE)
)

query.awaitTermination()